In [1]:
## 

In [2]:
import dagshub
dagshub.init(repo_owner='bteinstein',
             repo_name='demand_engine',
             mlflow=True)

Accessing as bteinstein

Initialized MLflow to track repo "bteinstein/demand_engine"

Repository bteinstein/demand_engine initialized!

# Demand Engine

In [3]:
import sys
import os

# Add the root directory to the Python path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)


import pandas as pd
import numpy as np
from datetime import datetime
from config import COLD_START_VALUE
from features.feature_engineering import SKUPurchaseFeatureEngineer, AdvancedNegativeSampler 
from features.feature_engineering import TrainingDataBuilder, generate_next_week_predictions
# from features.negative_sampling_dep import generate_negative_samples
from models.train_model import train_model
from predict.predict_new_week import predict_for_new_week
from evaluate.evaluate_model import evaluate_model
from monitor.drift_detection import detect_drift
import joblib

## Load Data

In [4]:
# Load data
transactions = pd.read_csv("../data/raw/transactions.csv")
sku_metadata = pd.read_csv("../data/raw/sku_metadata.csv")
customer_metadata = pd.read_csv("../data/raw/customer_metadata.csv")

In [5]:
tot_weeks = transactions.Week.nunique()
# We will use the last 5% of weeks for testing
cutoff_week_num =  max(1, int(tot_weeks * 0.02))
# Cutoff week for training
cutoff_week = pd.to_datetime(transactions.Week).max() - pd.Timedelta(weeks=cutoff_week_num) 
print(f'''
      total weeks: {tot_weeks}
      cutoff week number: {cutoff_week_num}
      min week: {transactions.Week.min()},
      max week: {transactions.Week.max()}
      cutoff week: {cutoff_week}
      ''')


      total weeks: 105
      cutoff week number: 2
      min week: 2022-01-02,
      max week: 2023-12-31
      cutoff week: 2023-12-17 00:00:00
      


In [6]:
# # prediction_week = None
# transactions.head()
# # sku_metadata.info()
# # customer_metadata.info()

# for dat in [transactions, sku_metadata, customer_metadata]:
#     print(dat.columns)

## Training Data

In [7]:
# Initialize components
fe = SKUPurchaseFeatureEngineer(cold_start_value=9999, rolling_windows = [4, 8, 12])
ns = AdvancedNegativeSampler(random_state=42)
builder = TrainingDataBuilder(fe, ns)


In [8]:
# Build training data
training_data, feature_cols = builder.build_training_dataset(
    transactions=transactions,
    sku_metadata=sku_metadata,
    customer_metadata=customer_metadata,
    prediction_week=cutoff_week, # "2024-01-01"
    max_negatives=5,
    category_weight=0.5,
    trending_weight=0.3
)

Generating comprehensive features...
Generating advanced negative samples...
Filling missing values...


In [14]:
print(training_data.shape[0])
print(training_data.drop_duplicates().shape[0])

15378
10505


In [10]:
non_string_values = training_data['Category'].apply(lambda x: not isinstance(x, str))
print(training_data['Category'][non_string_values])

Series([], Name: Category, dtype: object)


In [11]:
training_data.isnull().sum()

Week                                       0
CustomerID                                 0
SKUID                                      0
label                                      0
Category                                   0
                                          ..
Town_Category_PurchaseCount_12Weeks        0
Town_Segment_PurchaseCount_12Weeks         0
Town_SKU_PurchaseCount_12Weeks             0
Town_Manufacturer_PurchaseCount_12Weeks    0
prediction_week                            0
Length: 65, dtype: int64

In [ ]:
cols = ['CustomerID','SKUID']
training_data[cols] = training_data[cols].astype('str')

In [ ]:
## Save the training data to a file
# DATA_PATH = "data/processed/training_data.parquet" 
training_data.to_parquet("../data/processed/training_data.parquet", index=False)


In [ ]:
print(f'''
      total weeks: {tot_weeks}
      cutoff week number: {cutoff_week_num}
      min week: {transactions.Week.min()},
      max week: {transactions.Week.max()}
      cutoff week: {cutoff_week}
      ''')

print("Training Data Sample:")
print("Max Training Data Week", training_data.Week.max())
print(f"Data Size: {len(training_data):,} rows") # 10,505 ---> 15,378
print(training_data.label.value_counts())
print(training_data.label.value_counts(normalize=True))
print(sorted(feature_cols))
training_data.head()

# Training Data Sample:
# label
# 0    6713 (63%)
# 1    4070 (37%) 

In [ ]:
print(training_data.columns.tolist())
print(training_data.CustomerID.nunique()) 
print(training_data.SKUID.nunique())

## Inference Data - v1

In [ ]:
# Initialize your feature engineer
feature_engineer = SKUPurchaseFeatureEngineer()

# Generate inference dataset
inference_df, feature_columns = generate_next_week_predictions(
    transactions, sku_metadata, customer_metadata, feature_engineer
)

# inference_df now contains one row per (CustomerID, SKUID) with all engineered features
# Ready for model.predict(inference_df[feature_columns])

In [ ]:
print(inference_df.columns.tolist())
print(transactions.CustomerID.nunique())
print(inference_df.CustomerID.nunique())
print(inference_df.SKUID.nunique())

## Model Development

In [ ]:
# Set up MLflow tracking with DagsHub
# os.environ['MLFLOW_TRACKING_URI'] = "https://dagshub.com/bteinstein/demand_engine.mlflow"
# os.environ['MLFLOW_TRACKING_USERNAME'] = os.getenv("DAGSHUB_USER")
# os.environ['MLFLOW_TRACKING_PASSWORD'] = os.getenv("DAGSHUB_TOKEN")

In [ ]:
from models.train_models import prepare_data, evaluate_model

--------------------------------------------------------------------

## ERRATA

print

In [ ]:
# print(f"COLD_START_VALUE: {COLD_START_VALUE}")
# print(f"Number of rows in transactions data: {len(transactions)}")
# print(f"Number of rows in featured_data data: {len(featured_data)}")
# print(f"Number of rows in full data: {len(full_data)}")